# 37 — Validation–Leaderboard Postmortem
Analysis only: no model training, package/data download or submission creation. Public scores are restricted to repository-verified/user-supplied records. Correlations with n≤4 are labelled insufficient evidence.


## Predeclared diagnostic design
Validation–test similarity is target-free: mask 25%, lag completeness 25%, month 10%, latitude 8%, longitude 7%, location overlap 10%, climate regime 10%, date-gap 5%. These weights diagnose coverage and do not optimize public scores. The primary audit asks whether TWS availability was propagated before constructing lag features.


In [1]:
%run ../scripts/notebook37_runtime.py


Notebook37 artifacts written: C:\global-water-storage-forecasting\artifacts\notebook37


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display,Markdown
R=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'data/raw/Train.csv').exists());A=R/'artifacts/notebook37'
for n in ['submission_inventory','local_public_metric_inventory','validation_public_correlation','f1_l12_submission_difference_analysis','test_support_mismatch','validation_test_similarity','masking_pattern_audit','public_positive_negative_comparison','conservative_salvage_analysis']:
 display(Markdown('### '+n));display(pd.read_csv(A/f'{n}.csv').head(100))


### submission_inventory

,filename,origin_notebook,method,SHA256,row_count,known_count,masked_count,public_score,canonical_correlation,current_best_correlation,canonical_MAD,current_best_MAD,prediction_mean,prediction_std,prediction_min,prediction_max,changed_row_fraction,ID_aligned,status
0,submission_12_optimized.csv,not verified,historical/experimental; see origin notebook,11ce5fc87e2858e146c30e9fa8bddce90aa444f0413e7e...,280961,94048,186913,NaN,0.957027,0.954738,1.035622e-01,0.106536,-0.141210,0.483717,-2.179287,2.054586,0.676649,True,public unknown
1,submission_13_generalization.csv,not verified,historical/experimental; see origin notebook,b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737...,280961,94048,186913,NaN,1.000000,0.999513,0.000000e+00,0.006555,-0.130944,0.533372,-2.255716,2.054586,0.000000,True,public unknown
2,submission_14.csv,14,alternative-model blend,456fde8fca3262faf37acb12cf76610d350565a98655ee...,280961,94048,186913,0.825971,0.981455,0.981596,6.469922e-02,0.064334,-0.117317,0.513367,-2.255716,2.054586,0.678891,True,public-negative vs canonical
3,submission_15_primary.csv,15,delta dual expert,47335854a382a0578c2e6bb123c40a87874f2442f7c3f2...,280961,94048,186913,0.824798,0.974585,0.974152,8.976528e-02,0.090685,-0.123474,0.536520,-2.329112,2.123260,1.000000,True,public-negative vs canonical
4,submission_16_A_best_single.csv,not verified,historical/experimental; see origin notebook,0a44a22e90a9ba2952634b10ee658505fa0cda38c4f093...,280961,94048,186913,NaN,0.976819,0.976822,7.312285e-02,0.073127,-0.113644,0.540061,-2.255716,2.054586,0.665263,True,public unknown
5,submission_16_C_notebook11_diverse.csv,not verified,historical/experimental; see origin notebook,371ae2b2a0fae3e44f7b4660cbe45371e8662e2e6e20f5...,280961,94048,186913,NaN,0.991514,0.991447,4.592966e-02,0.045485,-0.112960,0.539840,-2.255716,2.054586,0.665263,True,public unknown
6,submission_17_notebook11_exact_reproduction.csv,not verified,historical/experimental; see origin notebook,b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737...,280961,94048,186913,NaN,1.000000,0.999513,0.000000e+00,0.006555,-0.130944,0.533372,-2.255716,2.054586,0.000000,True,public unknown
7,submission_18_A_domain_best.csv,18,drying-transition domain feature,9dc67a1d0abb31e3dbfadcf9758c23bf47c7bd74ad2c84...,280961,94048,186913,0.831108,0.988927,0.988838,5.026522e-02,0.050967,-0.130976,0.531513,-2.255716,2.095665,0.665263,True,public-negative vs canonical
8,submission_18_B_canonical_diverse.csv,not verified,historical/experimental; see origin notebook,e383849131bd7616074534644cfc8b377af5bd1b2b5a3f...,280961,94048,186913,NaN,1.000000,0.999513,6.164371e-20,0.006555,-0.130944,0.533372,-2.255716,2.054586,0.000157,True,public unknown
9,submission_21_spatial_residual_candidate.csv,21,capped southern masked correction,84d4ce6efdbfc3fbca55d9eea1ebccfe628ffa962e31b6...,280961,94048,186913,0.823609,0.999513,1.000000,6.554646e-03,0.000000,-0.124389,0.535866,-2.255716,2.054586,0.338424,True,public-positive vs canonical


### local_public_metric_inventory

,model,notebook,validation_design,overall_RMSE,known_RMSE,masked_RMSE,worst_fold,fold_std,spatial_RMSE,space_time_RMSE,public_score,public_delta_vs_canonical
0,Canonical,11,Notebook11 temporal,0.711647,NaN,0.758030,0.769866,0.060838,NaN,NaN,0.824464,0.000000
1,Alternative ensemble,14,Notebook11 recorded representative,0.688811,NaN,NaN,NaN,NaN,0.823676,0.804696,0.825971,0.001507
2,Hydrology feature,18,Notebook11 recorded representative,0.709586,NaN,NaN,NaN,NaN,0.853484,0.816142,0.831108,0.006644
3,Spatial correction,21,8-fold correction validation,0.757292,NaN,NaN,0.828759,0.058443,0.795834,NaN,0.823609,-0.000854
4,Graph E5 cluster,30,four representative robust folds,0.736826,0.633823,0.768143,1.154523,0.169696,0.853938,0.776147,NaN,NaN
5,ERA5 E1,31,three-region extended pilot,0.755780,0.755160,0.739141,0.843496,0.129217,NaN,NaN,NaN,NaN
6,GRU S1,33,latitude-stratified pilot strict_temporal_2,0.696528,0.594448,0.727309,0.696528,NaN,NaN,NaN,NaN,NaN
7,F1_L12 pilot,35,same Notebook33 pilot,0.640994,0.583242,0.659080,0.694233,NaN,NaN,NaN,0.824335,-0.000129
8,F1_L12 full,36,frozen full; mask propagation mismatch identif...,0.602115,0.567083,0.613352,0.635764,0.034667,0.671604,0.721241,0.824335,-0.000129


### validation_public_correlation

,metric,Pearson,Spearman,pairwise_ranking_accuracy,sign_agreement_vs_canonical,sample_count,evidence
0,Irregular hindcast,-0.197593,-0.8,0.166667,0.25,4,insufficient evidence: n=4; different historic...
1,Notebook 11 temporal,0.345023,-0.4,0.333333,0.25,4,insufficient evidence: n=4; different historic...
2,Space-time,0.059513,-0.8,0.166667,0.25,4,insufficient evidence: n=4; different historic...
3,Spatial block,0.810646,0.4,0.666667,0.75,4,insufficient evidence: n=4; different historic...
4,Strict date-grouped,0.252798,0.2,0.500000,0.50,4,insufficient evidence: n=4; different historic...
5,masked RMSE,NaN,NaN,NaN,NaN,3,insufficient/non-common validation design
6,known RMSE,NaN,NaN,NaN,NaN,2,insufficient/non-common validation design
7,worst fold,NaN,NaN,NaN,NaN,4,insufficient/non-common validation design
8,fold std,NaN,NaN,NaN,NaN,3,insufficient/non-common validation design
9,southern hemisphere RMSE,NaN,NaN,NaN,NaN,6,insufficient/non-common validation design


... (8 outputs left out)

### public_positive_negative_comparison

,filename,group,public_score,public_delta,canonical_correlation,changed_row_fraction,known_change_share,masked_change_share,southern_change_share,correction_MAD,temporal_smoothness_MAD,spatial_dispersion,prediction_variance,mean_risk_flags_on_changed,mean_lag_missing_on_changed
0,submission_14.csv,negative,0.825971,0.001507,0.981455,0.678891,0.040713,1.000000,0.679162,0.064699,0.298298,0.497405,0.263546,0.921229,0.440556
1,submission_15_primary.csv,negative,0.824798,0.000334,0.974585,1.000000,1.000000,1.000000,1.000000,0.089765,0.324729,0.519117,0.287854,0.874990,0.414354
2,submission_18_A_domain_best.csv,negative,0.831108,0.006644,0.988927,0.665263,0.000000,1.000000,0.665715,0.050265,0.307122,0.517313,0.282506,0.924157,0.442175
3,submission_21_spatial_residual_candidate.csv,positive,0.823609,-0.000854,0.999513,0.338424,0.000000,0.508707,0.665715,0.006555,0.303330,0.522304,0.287152,1.434237,0.439597
4,submission_36_f1_l12_primary.csv,positive,0.824335,-0.000129,0.766577,1.000000,1.000000,1.000000,1.000000,0.334773,0.332692,0.504588,0.308426,0.874990,0.414354
5,GROUP_MEAN,negative,0.827292,0.002829,0.981656,0.781385,0.346904,1.000000,0.781626,0.068243,0.310050,0.511278,0.277969,0.906792,0.432362
6,GROUP_MEAN,positive,0.823972,-0.000491,0.883045,0.669212,0.500000,0.754354,0.832857,0.170664,0.318011,0.513446,0.297789,1.154613,0.426976


### conservative_salvage_analysis

,F1_weight,current_best_weight,prediction_correlation_with_current_best,MAD_vs_current_best,changed_row_fraction,southern_MAD,masked_MAD,low_lag_completeness_MAD,local_OOF_result,public_risk_assessment
0,0.05,0.95,0.999434,0.016660,1.0,0.015025,0.019775,0.019401,not available for current-best vs F1 on common...,"lower perturbation than F1 alone, but no publi..."
1,0.10,0.90,0.997693,0.033320,1.0,0.030050,0.039549,0.038802,not available for current-best vs F1 on common...,"lower perturbation than F1 alone, but no publi..."
2,0.20,0.80,0.990477,0.066641,1.0,0.060100,0.079099,0.077603,not available for current-best vs F1 on common...,"lower perturbation than F1 alone, but no publi..."
3,0.30,0.70,0.978042,0.099961,1.0,0.090150,0.118648,0.116405,not available for current-best vs F1 on common...,"lower perturbation than F1 alone, but no publi..."


In [3]:
req=['submission_inventory.csv','local_public_metric_inventory.csv','validation_public_correlation.csv','f1_l12_submission_difference_analysis.csv','test_support_mismatch.csv','validation_test_similarity.csv','masking_pattern_audit.csv','public_positive_negative_comparison.csv','conservative_salvage_analysis.csv','validation_redesign.md','notebook38_plan.md','final_recommendation.md']
assert not [x for x in req if not (A/x).exists()]
assert not list((R/'submissions').glob('submission_37*.csv'))
display(Markdown((A/'validation_redesign.md').read_text(encoding='utf-8')));display(Markdown((A/'notebook38_plan.md').read_text(encoding='utf-8')));display(Markdown((A/'final_recommendation.md').read_text(encoding='utf-8')))


# Validation redesign

## Root cause found
Notebook 36 routed validation rows into known/masked experts **after** building the global TWS lag cache. A validation date labelled masked therefore still contributed its true `TWS_t` to later lag features. Irregular folds also allowed omitted intermediate Train dates into history. This is not direct target-label leakage, but it violates operational prediction-time availability and makes F1's masked validation materially easier than test inference.

## Keep / secondary / retire
- **Keep:** date-grouped temporal cutoffs, spatial blocks, row alignment assertions, worst-fold reporting and checkpointing.
- **Secondary:** irregular and spatial folds after rebuilding histories from an issue-time availability mask. Their current scores are diagnostic only.
- **Retire as decision metrics:** Notebook 36 F1 scores computed from unmasked global lag cache; `grid_asia` and `space_time_asia` as two independent signals (they are identical masks); pilot-only RMSE for submission promotion.

## Predeclared target-free decision score
After repairing masking, use 30% masking-pattern similarity, 20% lag-completeness similarity, 15% temporal/date-gap similarity, 15% spatial-distribution similarity, 10% worst-fold robustness and 10% location/climate coverage. Public score is excluded from the score. A candidate must also improve masked RMSE and pass every operational-availability assertion; the weighted score cannot compensate for a failed hard gate.


# Notebook 38 plan — test-matched masking validation

## Selected experiment
**B) Test-matched masking validation design.** No model or feature search.

1. Freeze canonical E0 and F1-L12 parameters/features.
2. For every fold, map the exact test date-offset and `TWS_t` availability pattern onto validation dates.
3. Apply the availability mask to the entire validation-origin panel **before** any `shift(k)` calculation.
4. Exclude non-issue-date intermediate observations in irregular simulations unless they would be operationally available.
5. Rebuild only TWS lag/missing-mask features; other covariate lags remain cutoff-safe.
6. Assert every TWS lag source was available under the simulated issue-time mask.
7. Compare E0/F1 on common rows. Report strict, masked, worst-fold and lag-completeness strata.

Acceptance: F1 must retain ≥0.002 overall and ≥0.004 masked gain, no negative space-time/spatial direction, and positive gain in a majority of folds. Stop and reject F1 if its gain collapses under propagated masking. Only after an independent rerun may a conservative current-best blend be considered; Notebook 38 itself should not use public score for weighting.


# Notebook 37 final recommendation

1. **Most likely cause:** validation masking was routed after lag construction. F1 validation histories retained TWS values from dates that the simulated/test availability pattern masks; irregular folds also consumed intermediate observations absent from the issue-date pattern. This operational mismatch inflates local lag value.
2. **Most public-associated metric:** `Spatial block` has the highest observed pairwise accuracy (0.667), but n=4 is insufficient; no metric is statistically established.
3. **Misleading metric:** Notebook 36 F1 strict/irregular/spatial/space-time RMSE in its present form, because all use the unpropagated global TWS lag cache. `grid_asia` and space-time are additionally identical.
4. **Masking simulated adequately?** No. See `masking_pattern_audit.csv`; expert routing matched current-row status but historical availability was not propagated into lag creation.
5. **Riskiest changes:** masked rows with incomplete real test history and large absolute F1–canonical corrections; support strata are quantified without target labels.
6. **Public-positive common behavior:** evidence is too small for a universal pattern. The only model beating both canonical and F1 is Notebook 21's conservative correction: exact file comparison differs on 33.8% of rows, but its cap is 0.05 and MAD is only 0.0066. F1 changed every row with MAD 0.3348 and did not beat current-best.
7. **Can low-weight blend salvage F1?** Plausible only as a hypothesis. Fixed blends sharply reduce perturbation, but no common corrected OOF evidence exists; no file was produced.
8. **Validation redesign:** apply test-matched TWS masks before lagging, remove unavailable intermediate history, retain hard temporal/spatial gates, and do not use public scores in the decision score.
9. **One next experiment:** B) test-matched masking validation design.
10. **Evidence required before submission:** reproducible positive F1 gain on propagated-mask OOF, masked gain ≥0.004, nonnegative spatial/space-time direction, majority-fold gain, complete availability audit, and an independent rerun.

Observed public facts: canonical 0.824463818, current-best 0.823609466, F1 0.824335279. F1 improved canonical by 0.000128539 but trailed current-best by 0.000725813. No new submission/model/package/data was created.
